# Object-Oriented Programming in C++ — CO1

**2310261L.CO.1** — *Develop solutions for real world problems using Object Oriented
Programming.* **[L3]**

---

## The problem

A processor's workings are invisible, so students memorise the fetch-decode-execute
diagram without ever seeing it. Our solution is a simulator you can step through — 4,102
lines across 28 files. Object orientation is what keeps it manageable.

![what each class owns](diagrams/oop-classes.png)

| Class | Owns | Responsible for |
|---|---|---|
| `RegisterFile` | the registers and flags | reads, writes, reporting what changed |
| `ALU` | **nothing** | operation + two values in, result + flags out |
| `Memory` | the store | reads and writes by address |
| `CPU` | all of the above, and the program | running one step |
| `Canvas` | the pixel buffer | coordinates into pixels |

---

## 1 · Fifteen instructions, one interface

```cpp
class Instruction {
public:
    virtual ~Instruction() {}
    virtual void           execute(CPU& cpu) = 0;
    virtual ControlSignals signals() const   = 0;
};

class StoreInstruction : public Instruction {
public:
    StoreInstruction(int rs, Word addr)
        : Instruction("STORE", -1, rs, addr, true) {}
    void           execute(CPU& cpu);
    ControlSignals signals() const;
};
```
<sub>src/core/Instruction.h:65</sub>

```cpp
void StoreInstruction::execute(CPU& cpu) {
    Word v = cpu.registers().readGP(rs_);
    cpu.registers().mar().write(operand_);
    cpu.registers().mdr().write(v);
    cpu.memory().write(operand_.raw(), v);
    cpu.setAluActivity(v, Word(0), v);
}

ControlSignals StoreInstruction::signals() const {
    ControlSignals s;
    s.marLoad   = true;
    s.memWrite  = true;
    s.regRead   = true;
    s.busActive = true;
    s.pcInc     = true;
    return s;
}
```
<sub>src/core/Instruction.cpp:126</sub>

And the processor's entire execute stage:

```cpp
current_->execute(*this);
```
<sub>src/core/CPU.cpp:163</sub>

**Adding a sixteenth instruction means one new class.** The CPU is not touched.

![branch chain against one class per instruction](diagrams/oop-complexity.png)

---

## 2 · Objects that free their own memory

```cpp
template <typename T>
class LinkedList {
    struct Node {
        T     data;
        Node* next;
        explicit Node(const T& d) : data(d), next(0) {}
    };
    Node*  head_;
    Node*  tail_;
    size_t size_;

public:
    LinkedList() : head_(0), tail_(0), size_(0) {}

    LinkedList(const LinkedList& other) : head_(0), tail_(0), size_(0) {
        for (Node* n = other.head_; n != 0; n = n->next) pushBack(n->data);
    }

    ~LinkedList() { clear(); }

    void clear() {
        Node* n = head_;
        while (n != 0) { Node* nx = n->next; delete n; n = nx; }
        head_ = tail_ = 0;
        size_ = 0;
    }
};
```
<sub>src/ds/LinkedList.h:15</sub>

Constructor, destructor and copy constructor together: **an object owns its memory for its
whole life.** The copy builds new nodes, so two lists never share one.

---

## 3 · Errors that report instead of crashing

```cpp
class SimulatorException : public std::exception {
protected:
    std::string message_;
public:
    explicit SimulatorException(const std::string& msg) : message_(msg) {}
    virtual const char* what() const throw() { return message_.c_str(); }
    virtual std::string kind() const { return "SimulatorException"; }
};

class InvalidOpcodeException : public SimulatorException {
public:
    explicit InvalidOpcodeException(const std::string& msg)
        : SimulatorException("Invalid opcode: " + msg) {}
    std::string kind() const { return "InvalidOpcodeException"; }
};
```
<sub>src/core/Exceptions.h:16</sub>

```cpp
catch (const SimulatorException& e) {
    std::cout << "  [" << e.kind() << "] " << e.what() << "\n";
}
```
<sub>src/main.cpp:342</sub>

Seven error types, one catch block:

```
[AssemblyErrorException] Assembly error: line 4: 'R9' is not a register
```

---

## 4 · Code that reads like what it describes

```cpp
// -- arithmetic / logic operators ---------------------------------------
Word operator+(const Word& o) const { return Word(static_cast<u16>(value_ + o.value_)); }
Word operator-(const Word& o) const { return Word(static_cast<u16>(value_ - o.value_)); }
Word operator&(const Word& o) const { return Word(static_cast<u16>(value_ & o.value_)); }
Word operator|(const Word& o) const { return Word(static_cast<u16>(value_ | o.value_)); }
Word operator^(const Word& o) const { return Word(static_cast<u16>(value_ ^ o.value_)); }
Word operator~()              const { return Word(static_cast<u16>(~value_)); }
Word operator<<(int n)        const { return Word(static_cast<u16>(value_ << n)); }
Word operator>>(int n)        const { return Word(static_cast<u16>(value_ >> n)); }

Word& operator++()    { ++value_; return *this; }              // pre-increment
Word  operator++(int) { Word t(*this); ++value_; return t; }   // post-increment
```
<sub>src/core/Word.h:44</sub>

`a + b`, not `add16(a, b)`.

---

## In one minute

1. **The problem:** a processor is invisible, so we built one you can step through.
2. **Each class owns its state.** The ALU owns *nothing* — which is why it can be tested
   alone.
3. **Fifteen instructions, one interface.** The whole execute stage is
   `current_->execute(*this)`.
4. **Objects free their own memory** — constructor, destructor, copy constructor.
5. **Errors report, not crash** — one hierarchy, one catch, naming the line.